# **Maternal Health Risk — Notebook 2 of 5: Preprocessing**

This notebook turns the raw CSV into clean train/test arrays ready for modelling.  
Steps: load → handle duplicates → handle outliers → encode target → scale features → train/test split → optional class-imbalance handling (SMOTE).


## **1. Imports & Setup**

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

RANDOM_STATE = 42
DATA_PATH = Path('..') / 'maternal_health_risk.csv'
ARTIFACTS = Path('artifacts'); ARTIFACTS.mkdir(exist_ok=True)


## **2. Load Raw Data**

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Raw shape:', df.shape)
df.head()


## **3. Handle Duplicate Rows**

**Decision.** The dataset contains many duplicate rows. We drop them so each unique reading contributes a single training observation, which prevents the model from memorising repeated patterns and is the more conservative academic choice. *(Keep them if your supervisor prefers — just skip this cell.)*


In [ ]:
print('Duplicate rows before:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print('Shape after dropping duplicates:', df.shape)


## **4. Handle Outliers**

We saw in EDA that `HeartRate` has values like **7 bpm** which are clinically impossible.  
We cap implausible values using domain rules instead of blind IQR clipping.


In [ ]:
# Domain rules for plausibility (adult heart rate ≥ 40 bpm, BS ≥ 1)
df.loc[df['HeartRate'] < 40, 'HeartRate'] = df['HeartRate'].median()
df.loc[df['BS']        < 1 , 'BS']        = df['BS'].median()
df.describe().T[['min','max','mean','std']]


## **5. Separate Features & Target / Encode the Target**

In [ ]:
X = df.drop(columns=['RiskLevel']).copy()
y_raw = df['RiskLevel'].copy()

le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = list(le.classes_)
print('Classes (encoded order):', class_names)


## **6. Train / Test Split (stratified)**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

print('Train:', X_train.shape, ' Test:', X_test.shape)
print('Train class distribution:', np.bincount(y_train))
print('Test  class distribution:', np.bincount(y_test))


## **7. Feature Scaling**

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

pd.DataFrame(X_train_sc, columns=X.columns).describe().T[['mean','std','min','max']]


## **8. (Optional) Handle Class Imbalance with SMOTE**

Only fit SMOTE on the **training** data — never the test set. Skip this cell if you want to compare imbalanced vs balanced runs.


In [ ]:
try:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)
    print('After SMOTE:', np.bincount(y_train_sm))
except ImportError:
    print('imbalanced-learn not installed — run: pip install imbalanced-learn')
    X_train_sm, y_train_sm = X_train_sc, y_train


## **9. Save Preprocessed Artifacts**

In [ ]:
joblib.dump({
    'X_train': X_train_sc,
    'X_test':  X_test_sc,
    'y_train': y_train,
    'y_test':  y_test,
    'X_train_sm': X_train_sm,
    'y_train_sm': y_train_sm,
    'feature_names': list(X.columns),
    'class_names':   class_names,
    'scaler':        scaler,
    'label_encoder': le,
}, ARTIFACTS/'preprocessed.joblib')
print('Saved:', ARTIFACTS/'preprocessed.joblib')


**Output of this notebook:** `artifacts/preprocessed.joblib` — loaded by notebooks **3** (models) and **4** (hyper-tuning).
